# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [37]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import StratifiedKFold



## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [38]:
#preparing data
df = pd.read_csv('../data/dayofweek.csv')


In [39]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=21, stratify=y_train
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [41]:
svm = SVC(
    C=10,
    kernel='rbf',
    gamma='auto',
    probability=True,
    random_state=21
)

svm.fit(X_train, y_train)

,C,10
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [42]:
dt = DecisionTreeClassifier(
    max_depth=25,
    criterion='entropy',
    class_weight='balanced',
    random_state=21
)

dt.fit(X_train, y_train)

,criterion,'entropy'
,splitter,'best'
,max_depth,25
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,21
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [43]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    criterion='entropy',
    random_state=21
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'entropy'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [44]:

def evaluate(model):
    y_pred = model.predict(X_valid)

    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')

    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")

In [45]:
print("FOR SVM:")
evaluate(svm)
print("FOR DT:")
evaluate(dt)
print("FOR RF:")
evaluate(rf)

FOR SVM:
accuracy is 0.70000
precision is 0.68713
recall is 0.70000
FOR DT:
accuracy is 0.85556
precision is 0.85888
recall is 0.85556
FOR RF:
accuracy is 0.89630
precision is 0.89615
recall is 0.89630


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [46]:
voting = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('dt', dt),
        ('rf', rf)
    ],
    voting='soft'
)
voting.fit(X_train, y_train)

,estimators,"[('svm', ...), ('dt', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,C,10
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0


In [47]:
evaluate(voting)

accuracy is 0.87407
precision is 0.87634
recall is 0.87407


In [48]:
voting = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('dt', dt),
        ('rf', rf)
    ],
    voting='soft',
    weights=[1,1,1]   # baseline
)

In [49]:
best_score = 0
best_weights = None

for w1 in [1,2]:
    for w2 in [1,2]:
        for w3 in [1,2]:
            voting = VotingClassifier(
                estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
                voting='soft',
                weights=[w1, w2, w3]
            )
            voting.fit(X_train, y_train)

            y_pred = voting.predict(X_valid)
            acc = accuracy_score(y_valid, y_pred)

            if acc > best_score:
                best_score = acc
                best_weights = [w1, w2, w3]

print(best_weights, best_score)

[2, 1, 2] 0.8851851851851852


In [50]:
voting = VotingClassifier(
    estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
    voting='soft',
    weights=best_weights
)

voting.fit(X_train, y_train)

y_pred = voting.predict(X_test)

print("accuracy is", accuracy_score(y_test, y_pred))
print("precision is", precision_score(y_test, y_pred, average='weighted'))
print("recall is", recall_score(y_test, y_pred, average='weighted'))

accuracy is 0.8964497041420119
precision is 0.8977372678413528
recall is 0.8964497041420119


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [51]:
bag = BaggingClassifier(
    estimator=svm,  
    n_estimators=10,
    random_state=21
)


In [52]:
best_score = 0
best_n = None

for n in [5, 10, 20, 50]:
    bag = BaggingClassifier(
        estimator=svm,
        n_estimators=n,
        max_samples=0.8,
        bootstrap=True,
        random_state=21
    )
    
    bag.fit(X_train, y_train)
    y_pred = bag.predict(X_valid)
    
    acc = accuracy_score(y_valid, y_pred)
    
    if acc > best_score:
        best_score = acc
        best_n = n

print(best_n, best_score)

50 0.7148148148148148


In [53]:
bag = BaggingClassifier(
    estimator=svm,
    n_estimators=best_n,
    random_state=21
)

bag.fit(X_train, y_train)

,estimator,"SVC(C=10, gam...ndom_state=21)"
,n_estimators,50
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,21
,verbose,0


In [54]:
y_pred = bag.predict(X_test)

print("accuracy is", accuracy_score(y_test, y_pred))
print("precision is", precision_score(y_test, y_pred, average='weighted'))
print("recall is", recall_score(y_test, y_pred, average='weighted'))

accuracy is 0.7603550295857988
precision is 0.7744354054590653
recall is 0.7603550295857988


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [55]:
best_score = 0
best_params = None
for n in [2, 3, 4, 5, 6, 7]:
    for pt in [True, False]:
        
        cv = StratifiedKFold(
            n_splits=n,
            shuffle=True,
            random_state=21
        )
        
        stack = StackingClassifier(
            estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv,
            passthrough=pt
        )
        
        stack.fit(X_train, y_train)
        y_pred = stack.predict(X_valid)
        
        acc = accuracy_score(y_valid, y_pred)
        
        if acc > best_score:
            best_score = acc
            best_params = (n, pt)

print(best_params, best_score)

c:\Users\Asror\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Asror\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Asror\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is depr

(6, True) 0.9


c:\Users\Asror\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


In [56]:
n, pt = best_params

cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)

stack = StackingClassifier(
    estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=cv,
    passthrough=pt
)

stack.fit(X_train, y_train)

c:\Users\Asror\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


,estimators,"[('svm', ...), ('dt', ...), ...]"
,final_estimator,LogisticRegre...r='liblinear')
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,None
,passthrough,True
,verbose,0
,C,10
,kernel,'rbf'
,degree,3
,gamma,'auto'


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [57]:
best_model = voting

In [58]:
y_pred = best_model.predict(X_test)

In [65]:
df_res = pd.DataFrame({
    'true': y_test,
    'pred': y_pred
})

errors = df_res[df_res['true'] != df_res['pred']]

weekday_error = (
    errors['true'].value_counts() /
    df_res['true'].value_counts()
) * 100

print(weekday_error.sort_values(ascending=False))

true
0    29.629630
5    14.814815
4    14.285714
2    10.000000
1     9.090909
6     5.633803
3     5.000000
Name: count, dtype: float64


In [60]:
weekday_errors = errors['true'].value_counts(normalize=True) * 100
print(weekday_errors)

true
5    22.857143
0    22.857143
1    14.285714
6    11.428571
3    11.428571
4     8.571429
2     8.571429
Name: proportion, dtype: float64


In [66]:
X_columns = df.drop(columns=['dayofweek']).columns
X_test_df = pd.DataFrame(X_test, columns=X_columns)
df_res['labname'] = X_test_df.filter(like='labname').idxmax(axis=1)

lab_errors = df_res[df_res['true'] != df_res['pred']]

lab_error_percent = (
    lab_errors['labname'].value_counts() /
    df_res['labname'].value_counts()
).fillna(0) * 100

print(lab_error_percent.sort_values(ascending=False))

labname
labname_lab03       100.000000
labname_lab03s      100.000000
labname_laba04       25.714286
labname_laba04s      24.000000
labname_lab05s       16.666667
labname_laba06       11.111111
labname_code_rvw      7.692308
labname_project1      7.526882
labname_laba06s       6.666667
labname_laba05        0.000000
Name: count, dtype: float64


In [68]:
import joblib

joblib.dump(best_model, '../data/best_model_for_ex03.pkl')

['../data/best_model_for_ex03.pkl']